In [4]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments
from datasets import load_dataset
import numpy as np
import evaluate

In [ ]:
data=load_dataset('stanfordnlp/imdb')

In [6]:
train_data=data['train'].shuffle(seed=42).select(range(3000))
test_data=data['test'].shuffle(seed=42).select(range(3000))

In [ ]:
model_name="distilbert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)

In [8]:
def tokenize(data):
    result=tokenizer(data['text'],padding='max_length',truncation=True,max_length=256)
    return result

In [ ]:
tokenize_train=train_data.map(tokenize,batched=True)
tokenize_test=test_data.map(tokenize,batched=True)

In [ ]:
accuracy=evaluate.load("accuracy")

In [11]:
def metrics(result):
  predictions=np.argmax(result.predictions,axis=-1)
  accur=accuracy.compute(predictions=predictions,references=result.label_ids)
  return accur

In [12]:
training_args=TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch"
)

In [13]:
trainer=Trainer(model=model,args=training_args,train_dataset=tokenize_train,eval_dataset=tokenize_test,compute_metrics=metrics)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [28]:
from sklearn.metrics import classification_report
predictions = trainer.predict(tokenize_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
print(classification_report(labels, preds, target_names=["negative", "positive"]))


              precision    recall  f1-score   support

    negative       0.89      0.87      0.88      1511
    positive       0.87      0.89      0.88      1489

    accuracy                           0.88      3000
   macro avg       0.88      0.88      0.88      3000
weighted avg       0.88      0.88      0.88      3000



In [ ]:
model_2=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)

In [24]:
training2_args=TrainingArguments(output_dir="./results2",num_train_epochs=3,per_device_train_batch_size=16,per_device_eval_batch_size=16,eval_strategy="epoch")

In [25]:
trainer2=Trainer(model=model_2,args=training2_args,train_dataset=tokenize_train,eval_dataset=tokenize_test,compute_metrics=metrics)

In [ ]:
trainer2.train()

In [ ]:
trainer2.evaluate()

In [29]:
from sklearn.metrics import classification_report
predictions = trainer2.predict(tokenize_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
print(classification_report(labels, preds, target_names=["negative", "positive"]))


              precision    recall  f1-score   support

    negative       0.91      0.87      0.89      1511
    positive       0.87      0.91      0.89      1489

    accuracy                           0.89      3000
   macro avg       0.89      0.89      0.89      3000
weighted avg       0.89      0.89      0.89      3000

